In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder
import pytorch_lightning as pl

class CryptoDataset(Dataset):
    def __init__(self, features: pd.DataFrame, labels: pd.DataFrame):
        """
        Args:
            features (pd.DataFrame): DataFrame containing the preprocessed feature data.
            labels (pd.DataFrame): DataFrame containing the unprocessed labels.
        """
        self.features = features.values  # Convert features to NumPy array
        self.labels = labels['&-class']  # Extract the labels column

        # Encode the labels (e.g., 'up'/'down') into integers
        self.label_encoder = LabelEncoder()
        self.labels_encoded = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        # Return the total number of samples in the dataset
        return len(self.features)

    def __getitem__(self, idx):
        """
        Returns a single sample from the dataset, indexed by `idx`.
        """
        feature = torch.tensor(self.features[idx], dtype=torch.float32)
        label = torch.tensor(self.labels_encoded[idx], dtype=torch.long)
        return feature, label

class CryptoDataModule(pl.LightningDataModule):
    def __init__(self, directory_path, batch_size=64, train_split=0.8):
        super().__init__()
        self.directory_path = directory_path
        self.batch_size = batch_size
        self.train_split = train_split

    def setup(self, stage=None):
        """
        Load data and split into training and validation datasets.
        """
        features_path = os.path.join(self.directory_path, 'features_filtered.parquet')
        labels_path = os.path.join(self.directory_path, 'labels_filtered.parquet')

        print("Loading parquet files...")
        df_features = pd.read_parquet(features_path)
        df_labels = pd.read_parquet(labels_path)
        print("Parquet files loaded successfully.")

        # Create dataset
        dataset = CryptoDataset(features=df_features, labels=df_labels)

        # Split dataset into training and validation sets
        train_size = int(self.train_split * len(dataset))
        val_size = len(dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

# Main execution
if __name__ == "__main__":
    # Directory path
    directory_path = '/allah/data/parquet'

    # Initialize DataModule
    data_module = CryptoDataModule(directory_path=directory_path, batch_size=64)

    # Setup datasets
    data_module.setup()

    # Test train DataLoader
    train_loader = data_module.train_dataloader()
    for batch_features, batch_labels in train_loader:
        print(f"Features batch shape: {batch_features.shape}")
        print(f"Labels batch shape: {batch_labels.shape}")
        break

    # Test validation DataLoader
    val_loader = data_module.val_dataloader()
    for batch_features, batch_labels in val_loader:
        print(f"Validation Features batch shape: {batch_features.shape}")
        print(f"Validation Labels batch shape: {batch_labels.shape}")
        break


In [2]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F

class CryptoPricePredictor(pl.LightningModule):
    def __init__(self, input_dim, hidden_dim=64, output_dim=2, dropout_rate=0.1):
        super(CryptoPricePredictor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # First hidden layer
            nn.BatchNorm1d(hidden_dim),         # Batch normalization for better training stability
            nn.ReLU(),                          # Activation function
            nn.Dropout(dropout_rate),           # Dropout for regularization

            nn.Linear(hidden_dim, hidden_dim),  # Second hidden layer
            nn.BatchNorm1d(hidden_dim),         # Batch normalization
            nn.ReLU(),                          # Activation function
            nn.Dropout(dropout_rate),           # Dropout

            nn.Linear(hidden_dim, output_dim)   # Output layer
        )

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        features, labels = batch
        outputs = self(features)
        loss = F.cross_entropy(outputs, labels)
        
        # Training Accuracy
        acc = (outputs.argmax(dim=1) == labels).float().mean()
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_acc', acc, on_step=True, on_epoch=True, prog_bar=True)
        
        # Precision for binary classification
        preds = outputs.argmax(dim=1)
        tp = ((preds == 1) & (labels == 1)).float().sum()  # True Positives
        fp = ((preds == 1) & (labels == 0)).float().sum()  # False Positives
        precision = tp / (tp + fp + 1e-8)  # Adding epsilon to avoid division by zero
        self.log('train_precision', precision, on_step=True, on_epoch=True, prog_bar=True)
        
        return loss

    def validation_step(self, batch, batch_idx):
        features, labels = batch
        outputs = self(features)
        loss = F.cross_entropy(outputs, labels)
        
        # Validation Accuracy
        acc = (outputs.argmax(dim=1) == labels).float().mean()
        self.log('val_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('val_acc', acc, on_step=True, on_epoch=True, prog_bar=True)
        
        # Precision for binary classification
        preds = outputs.argmax(dim=1)
        tp = ((preds == 1) & (labels == 1)).float().sum()  # True Positives
        fp = ((preds == 1) & (labels == 0)).float().sum()  # False Positives
        precision = tp / (tp + fp + 1e-8)  # Adding epsilon to avoid division by zero
        self.log('val_precision', precision, on_step=True, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)


In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping

# Define input dimensions based on the features
input_dim = 86  # Replace with the actual number of features in your dataset

# Initialize the model
model = CryptoPricePredictor(input_dim=input_dim, hidden_dim=64, output_dim=2)

# Initialize the data module
directory_path = '/allah/data/parquet'  # Update with your actual directory path
data_module = CryptoDataModule(directory_path=directory_path, batch_size=64)

# Initialize EarlyStopping callback
early_stopping = EarlyStopping(
    monitor="val_loss",  # Metric to monitor
    mode="min",          # "min" for minimizing (e.g., loss), "max" for maximizing (e.g., accuracy)
    patience=7,          # Number of epochs with no improvement before stopping
    verbose=True         # Print messages when stopping
)

# Initialize the Trainer with the EarlyStopping callback
trainer = Trainer(max_epochs=1000, callbacks=[early_stopping])

# Train the model
trainer.fit(model, datamodule=data_module)


In [ ]:
!ls /allah/stuff/freq/userdir/backtest_results

In [1]:
# read backtest-result-2024-12-14_11-37-27.json
# backtest-result-2024-12-14_11-37-27.meta.json
# backtest-result-2024-12-14_11-37-27_market_change.feather

import json
import pandas as pd
with open('/allah/stuff/freq/userdir/backtest_results/backtest-result-2024-12-14_16-48-36.json', 'r') as f:
    long_backtest_results = json.load(f)
# Load the backtest results
with open('/allah/stuff/freq/userdir/backtest_results/backtest-result-2024-12-14_18-19-02.json', 'r') as f:
    short_backtest_results = json.load(f)


long_trades = long_backtest_results["strategy"]["TEMA50TrailingStopStrategy"]["trades"]
short_trades = short_backtest_results["strategy"]["TEMA50TrailingStopStrategy"]["trades"]

df_trades_long = pd.DataFrame(long_trades)
df_trades_short = pd.DataFrame(short_trades)

In [2]:
df_trades_long[['enter_tag', 'is_short', 'profit_abs', 'open_date']]
df_trades_long['is_long_profit'] = df_trades_long['profit_abs'] > 0 
df_trades_short[['enter_tag', 'is_short', 'profit_abs', 'open_date']]

df_trades_long['is_long_profit'] = df_trades_long['profit_abs'] > 0 
df_trades_long['long_profitbility'] = df_trades_long['profit_abs']

df_trades_short['is_short_profit'] = df_trades_short['profit_abs'] > 0
df_trades_short['short_profitbility'] = df_trades_short['profit_abs']


In [ ]:
import pandas as pd

# Add the columns from df_trades_short to df_trades_long
df_trades_long['is_short_profit'] = df_trades_short['is_short_profit']
df_trades_long['short_profitability'] = df_trades_short['short_profitbility']

# Create a new DataFrame with only the specified columns
final_df = df_trades_long[['open_date', 'is_long_profit', 'long_profitability', 'is_short_profit', 'short_profitability']]


In [2]:
import pandas as pd

# Assuming df_trades_long and df_trades_short are already loaded with your data

# Fixing the typo in column names for clarity
df_trades_long['is_long_profit'] = df_trades_long['profit_abs'] > 0
df_trades_long['long_profitability'] = df_trades_long['profit_abs']

df_trades_short['is_short_profit'] = df_trades_short['profit_abs'] > 0
df_trades_short['short_profitability'] = df_trades_short['profit_abs']

# Merging the DataFrames based on a common column 'open_date' if it's unique or an appropriate key
# Ensure that 'open_date' is a suitable key for merging; if not, adjust accordingly
df_combined = pd.merge(df_trades_long, df_trades_short[['open_date', 'is_short_profit', 'short_profitability']],
                       on='open_date', how='left')

# Selecting only the required columns for the final DataFrame
final_df = df_combined[['open_date', 'is_long_profit', 'long_profitability', 'is_short_profit', 'short_profitability']]


In [3]:
final_df

,open_date,is_long_profit,long_profitability,is_short_profit,short_profitability
0,2024-01-01 04:57:00+00:00,False,-0.114344,False,-0.259773
1,2024-01-01 05:48:00+00:00,False,-0.111685,False,-0.179242
2,2024-01-01 06:06:00+00:00,False,-0.225260,False,-0.176343
3,2024-01-01 06:09:00+00:00,False,-0.159006,False,-0.243820
4,2024-01-01 06:12:00+00:00,False,-0.184820,False,-0.217173
...,...,...,...,...,...
19963,2024-10-31 22:54:00+00:00,False,-0.048318,False,-0.420046
19964,2024-10-31 22:57:00+00:00,False,-0.241854,False,-0.226703
19965,2024-10-31 23:06:00+00:00,False,-0.131043,False,-0.329608
19966,2024-10-31 23:15:00+00:00,False,-0.208705,True,0.008278


In [4]:
# Assuming final_df is loaded with your data
# Calculate the counts for each combination
count_both_false = len(final_df[(final_df['is_long_profit'] == False) & (final_df['is_short_profit'] == False)])
count_both_true = len(final_df[(final_df['is_long_profit'] == True) & (final_df['is_short_profit'] == True)])
count_long_true_short_false = len(final_df[(final_df['is_long_profit'] == True) & (final_df['is_short_profit'] == False)])
count_long_false_short_true = len(final_df[(final_df['is_long_profit'] == False) & (final_df['is_short_profit'] == True)])

# Print out the results
print("Both long and short profit False:", count_both_false)
print("Both long and short profit True:", count_both_true)
print("Long profit True and short profit False:", count_long_true_short_false)
print("Long profit False and short profit True:", count_long_false_short_true)


Both long and short profit False: 10063
Both long and short profit True: 0
Long profit True and short profit False: 4957
Long profit False and short profit True: 4948


In [ ]:
df_trades_long['long_profitability']

In [ ]:
df_trades_short[['enter_tag', 'is_short', 'profit_abs', 'open_date']]